# Baliser le corpus avec Stanza

In [ ]:
import stanza
import re
from pathlib import Path
from lxml import etree
from joblib import Parallel, delayed

In [23]:
# Téléchargement des modèles Stanza pour le français et l'italien
# À n'exécuter qu'une seule fois
stanza.download('fr', model_dir='C:/stanza_models')
stanza.download('it', model_dir='C:/stanza_models')

2026-03-19 12:06:10 INFO: Downloaded file to C:/stanza_models\resources.json
2026-03-19 12:06:10 INFO: Downloading default packages for language: fr (French) ...
2026-03-19 12:06:10 INFO: File exists: C:/stanza_models\fr\default.zip
2026-03-19 12:06:12 INFO: Finished downloading models and saved to C:/stanza_models


2026-03-19 12:06:12 INFO: Downloaded file to C:/stanza_models\resources.json
2026-03-19 12:06:12 INFO: Downloading default packages for language: it (Italian) ...
2026-03-19 12:06:13 INFO: File exists: C:/stanza_models\it\default.zip
2026-03-19 12:06:16 INFO: Finished downloading models and saved to C:/stanza_models


[['zip', 'default.zip']]

In [24]:
DIR_V0 = Path('output/v0/')
OUTPUT_DIR = Path('output/Vstanza/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Correspondance entre les labels Stanza et les balises TEI
LABEL_MAP = {
    "PER":  "persName",
    "LOC":  "placeName",
    "GPE":  "placeName",
}

In [28]:
from stanza.pipeline.core import DownloadMethod

nlp_fra = stanza.Pipeline(
    'fr',
    model_dir='C:/stanza_models',
    processors='tokenize,ner',
    use_gpu=False,
    download_method=DownloadMethod.REUSE_RESOURCES  # ← désactive la vérification réseau
)
nlp_ita = stanza.Pipeline(
    'it',
    model_dir='C:/stanza_models',
    processors='tokenize,ner',
    use_gpu=False,
    download_method=DownloadMethod.REUSE_RESOURCES
)

2026-03-19 12:08:05 WARNING: Language fr package default expects mwt, which has been added
2026-03-19 12:08:05 INFO: Loading these models for language: fr (French):
| Processor | Package            |
----------------------------------
| tokenize  | combined           |
| mwt       | combined           |
| ner       | wikinergold_charlm |

2026-03-19 12:08:05 INFO: Using device: cpu
2026-03-19 12:08:05 INFO: Loading: tokenize
2026-03-19 12:08:05 INFO: Loading: mwt
2026-03-19 12:08:05 INFO: Loading: ner
2026-03-19 12:08:09 INFO: Done loading processors!
2026-03-19 12:08:09 WARNING: Language it package default expects mwt, which has been added
2026-03-19 12:08:09 INFO: Loading these models for language: it (Italian):
| Processor | Package  |
------------------------
| tokenize  | combined |
| mwt       | combined |
| ner       | fbk      |

2026-03-19 12:08:09 INFO: Using device: cpu
2026-03-19 12:08:09 INFO: Loading: tokenize
2026-03-19 12:08:09 INFO: Loading: mwt
2026-03-19 12:08:09 INF

In [29]:
def detect_lang(text):
    """Détection heuristique de la langue (fr vs it) par marqueurs lexicaux."""
    fr_markers = len(re.findall(r'\b(le|la|les|de|du|des|un|une|et|en|je|il|elle|nous|vous|ils)\b', text, re.IGNORECASE))
    it_markers = len(re.findall(r'\b(il|la|le|di|del|della|un|una|e|in|io|lui|lei|noi|voi|loro)\b', text, re.IGNORECASE))
    return "it" if it_markers > fr_markers else "fr"


def get_entities(text, nlp_fra, nlp_ita):
    """
    Retourne la liste des entités détectées par Stanza sous forme de
    tuples (start_char, end_char, tag) triés par position.
    Seules les entités dont le label est dans LABEL_MAP sont conservées.
    """
    if not text or not text.strip():
        return []
    nlp = nlp_ita if detect_lang(text) == "it" else nlp_fra
    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        tag = LABEL_MAP.get(ent.type)
        if tag:
            entities.append((ent.start_char, ent.end_char, tag))
    return entities


def inject_entities(parent, original_text, entities, insert_before_index):
    """
    Remplace `original_text` (qui peut être element.text ou child.tail)
    par une suite de nœuds texte + éléments <persName>/<placeName>,
    en insérant les nouveaux éléments à la position `insert_before_index`
    dans `parent`.

    Principe conservatif :
    - On découpe `original_text` uniquement aux offsets exacts fournis par Stanza.
    - Chaque morceau non-entité reste une chaîne de caractères brute
      (text ou tail), sans aucune modification.
    - Chaque morceau entité devient un élément XML avec .text = substring exacte.

    Retourne le nombre d'éléments insérés.
    """
    if not entities:
        return 0

    # Construire la liste ordonnée de segments :
    # chaque segment est soit (str, None) soit (str, tag)
    segments = []
    last = 0
    for start, end, tag in entities:
        if start > last:
            segments.append((original_text[last:start], None))
        segments.append((original_text[start:end], tag))
        last = end
    if last < len(original_text):
        segments.append((original_text[last:], None))

    # ------------------------------------------------------------------
    # Injection dans l'arbre XML
    # ------------------------------------------------------------------
    # Le texte AVANT le premier élément inséré doit aller dans :
    #   - parent.text  si insert_before_index == 0
    #   - parent[insert_before_index - 1].tail  sinon
    # Le texte APRÈS chaque élément inséré va dans son .tail.
    # ------------------------------------------------------------------

    inserted = 0
    pending_text = ""   # texte à placer avant le prochain élément

    # Vider la source originale
    if insert_before_index == 0:
        parent.text = ""
    else:
        parent[insert_before_index - 1].tail = ""

    for text_chunk, tag in segments:
        if tag is None:
            # Texte brut : accumuler
            pending_text += text_chunk
        else:
            # Entité : on place d'abord le texte accumulé, puis l'élément
            pos = insert_before_index + inserted
            if pos == 0:
                parent.text = (parent.text or "") + pending_text
            else:
                prev = parent[pos - 1]
                prev.tail = (prev.tail or "") + pending_text
            pending_text = ""

            # Créer et insérer l'élément
            new_el = etree.Element(tag)
            new_el.text = text_chunk   # substring EXACTE du texte original
            new_el.tail = ""           # sera complété par le prochain pending_text
            parent.insert(pos, new_el)
            inserted += 1

    # Texte résiduel après le dernier élément (ou si aucune entité insérée)
    if pending_text:
        pos = insert_before_index + inserted
        if pos == 0:
            parent.text = (parent.text or "") + pending_text
        else:
            prev = parent[pos - 1]
            prev.tail = (prev.tail or "") + pending_text

    return inserted


def process_element(root, nlp_fra, nlp_ita):
    """
    Parcours itératif de l'arbre XML.
    Pour chaque nœud, on traite :
      - element.text  (texte avant le premier enfant)
      - child.tail    (texte après chaque enfant)
    On ne touche jamais aux attributs, aux balises existantes,
    ni au texte déjà balisé.
    """
    all_elements = []
    stack = [root]
    while stack:
        el = stack.pop()
        all_elements.append(el)
        for child in reversed(list(el)):
            stack.append(child)

    for element in all_elements:
        try:
            # --- Traitement de element.text ---
            if element.text and element.text.strip():
                entities = get_entities(element.text, nlp_fra, nlp_ita)
                if entities:
                    inject_entities(element, element.text, entities, insert_before_index=0)
                # Sinon : element.text est laissé absolument intact

            # --- Traitement des child.tail ---
            i = 0
            while i < len(element):
                child = element[i]
                i += 1
                if child.tail and child.tail.strip():
                    entities = get_entities(child.tail, nlp_fra, nlp_ita)
                    if entities:
                        tail_text = child.tail
                        child.tail = None   # sera recréé par inject_entities
                        inserted = inject_entities(element, tail_text, entities, insert_before_index=i)
                        i += inserted
                    # Sinon : child.tail est laissé absolument intact

        except Exception as e:
            import traceback
            print(f"  → Erreur sur élément <{element.tag}> : {e}")
            traceback.print_exc()
            continue


def process_file(xml_path, output_dir):
    """
    Traite un fichier XML dans son propre worker.
    Les pipelines Stanza sont instanciés localement (non sérialisables).
    """
    nlp_fra = stanza.Pipeline('fr', model_dir='C:/stanza_models', processors='tokenize,ner',
                              use_gpu=False, verbose=False,
                              download_method=DownloadMethod.REUSE_RESOURCES)
    nlp_ita = stanza.Pipeline('it', model_dir='C:/stanza_models', processors='tokenize,ner',
                              use_gpu=False, verbose=False,
                              download_method=DownloadMethod.REUSE_RESOURCES)
    try:
        tree = etree.parse(str(xml_path), etree.XMLParser(remove_blank_text=False))
        process_element(tree.getroot(), nlp_fra, nlp_ita)
        output_path = output_dir / xml_path.name
        tree.write(str(output_path), encoding="utf-8", xml_declaration=True, pretty_print=True)
        return f"✓ {xml_path.name}"
    except Exception as e:
        return f"✗ {xml_path.name} — erreur : {e}"


In [30]:
# --- Exécution parallèle ---
# Stanza est gourmand en mémoire : réduire n_jobs si nécessaire
xml_files = list(DIR_V0.glob('*.xml'))
print(f"{len(xml_files)} fichiers à traiter avec 4 workers...\n")

results = Parallel(n_jobs=4, backend="loky", verbose=0)(
    delayed(process_file)(f, OUTPUT_DIR) for f in xml_files
)

for r in results:
    print(r)
print("\nTerminé.")

19 fichiers à traiter avec 4 workers...



c:\Users\ebondoer\anaconda3\envs\stanza-env\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


✓ Agucchi_TrattatoPittura.xml
✓ Daret_VieRaphael.xml
✓ DupuyDuGrez_TraitePeinture.xml
✓ Freart_IdeaDellaPerfezione.xml
✓ Lomazzo_Idea.xml
✓ Lomazzo_TraicteProportion.xml
✓ Marino_DicerieSacre.xml
✓ Monier_HistoireArtsRapportDessein.xml
✓ Pader_LaPeintureParlante.xml
✓ Pader_SongeEnigmatique.xml
✓ Piles_AbregeViePeintres.xml
✓ Piles_ConversationsConnaissancePeinture.xml
✓ Piles_CoursPeinture.xml
✓ Piles_DialogueColoris.xml
✓ Vinci_TraitePeinture_fra.xml
✓ Vinci_TrattatoPittura_ITA.xml
✓ Zuccari_IdeaPittori.xml
✓ Zuccari_Lettera.xml
✓ Zuccari_OrigineProgressoAcademiaDissegno.xml

Terminé.
